# Setup

In [ ]:
from pathlib import Path

import numpy as np

import polars as pl
import polars.selectors as cs

import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.multivariate.factor import Factor
from sklearn.decomposition import PCA

from climate_attitudes.settings import Config
from climate_attitudes.dataset import Dataset
from climate_attitudes.correlation import Correlation, filter_vars_by_abs_corr
from climate_attitudes import configure_mpl
from climate_attitudes.visualisation import plot_corr, plot_corr_with_dendro
from climate_attitudes.parallel_analysis import pa_random_eigs, pa_true_eigs

FONT_PATH = Path("../fonts")
configure_mpl(FONT_PATH)

plt.rc("figure", dpi=150)

# Select survey questions

Excluded columns we'd like to include:

- ccGovt, ccIO: Treatments
- ccComp, ccSolve
- cc13
- Media questions?

In [ ]:
SURVEY_COLS = [
    "participant_id",
    # "participant_type",
    "wave",
    # "start_date",
]

BELIEF_COLS = [
    "cc1",
    "cc4_world",
    "cc4_wealthUS",
    "cc4_poorUS",
    "cc4_comm",
    "cc5_world",
    "cc5_wealthUS",
    "cc5_poorUS",
    "cc5_comm",
    "cc10",
    "cc11",
    "cc12",
    "cvcc4_will",
    "soc_trust",
    "soc_help",
    "future",
    "cvcc10_cc",
]

EXPERIENCE_COLS = []

ATTITUDE_COLS = [
    "cc3",
    "cc6",
    # "ccIO",
    # "ccGovt",
    "cvcc_worryothers",
    "ew5",
    "cvcc4_should",
    "cvcc6",
    "cvcc9_econ",
    "cvcc9_cc",
    "cvcc9_cv",
    "cc_ica",
    "cc_pol_tax",
    "cc_pol_car",
    "pol4",
    "pol7",
    "pol7_pi",
    "pol8",
    "pol8_pi",
    "pol9",
    "pol10",
    "pol11",
    "pol11_pi",
    "pol_affiliation",
    "pol_ideology",
    "pol_worry_econ",
    "pol_trust_state",
    "pol_trust_io",
    "pol_trust_cdc",
    "pol_trust_news",
    "pol_trust_epa",
    "pol_trust_sci",
    "pol_laws_cong",
    "pol_laws_state",
    # "pol_vote_support",
]

BEHAVIOUR_COLS = [
    "ew6",
    "cvcc4_personal",
]

DEMOGRAPHIC_COLS = [
    # "dem_age",
    "dem_income",
    "dem_urban",
]

TREATMENT_COLUMNS = [
    # "Group_pol_vote_support",
]

REVERSE_CODING = [
    "cc10",
    "pol4",
    "pol8",
    "pol8_pi",
    "pol9",
    "pol10",
    "pol11",
    "pol11_pi",
    "dem_urban",
]

TRANSFORMS = [
    pl.col("cc1").replace({1: 2, 99: 1}),  # Move "yes" to 2, "don't know" to 1
    pl.col(r"^cc4_(world|wealthUS|poorUS|comm)$").replace(
        {1: 0, 2: 1, 99: 2}
    ),  # Shift "not at all", "only a little" down; insert "don't know" between "only a little" and "a moderate amount"
    pl.col(r"^cc5_(world|wealthUS|poorUS|comm)$").replace({1: 0, 2: 1, 99: 2}),
]

QUESTION_COLS = (
    BELIEF_COLS + EXPERIENCE_COLS + ATTITUDE_COLS + BEHAVIOUR_COLS + DEMOGRAPHIC_COLS
)

ALL_COLS = SURVEY_COLS + QUESTION_COLS + TREATMENT_COLUMNS

In [ ]:
categories = (
    ["Belief"] * len(BELIEF_COLS)
    + ["Experience"] * len(EXPERIENCE_COLS)
    + ["Attitude"] * len(ATTITUDE_COLS)
    + ["Behaviour"] * len(BEHAVIOUR_COLS)
    + ["Demographic"] * len(DEMOGRAPHIC_COLS)
)

# Load data

In [ ]:
config = Config(_env_file="../.env")

dataset = (
    Dataset.load(config)
    .filter_columns(ALL_COLS)
    .filter_at_least_one_resp(QUESTION_COLS)
    .cast_enum_to_int()
    .transform(*TRANSFORMS)
    .reverse_coding(REVERSE_CODING)
    .impute_viterbi(QUESTION_COLS)
)
dataset_std = dataset.standardise(cs.exclude(*SURVEY_COLS))

resp = dataset.response.collect()
resp_std = dataset_std.response.collect()

In [ ]:
sns.displot(
    resp.drop("participant_id", "wave")
    .unpivot(variable_name="column", value_name="response")
    .to_pandas(),
    x="response",
    col="column",
    col_wrap=4,
    discrete=True,
    common_norm=False,
    stat="probability",
    facet_kws=dict(sharex=False, sharey=False),
    aspect=2,
    height=1.5,
    shrink=0.7,
    common_bins=False,
)

In [ ]:
plot_corr_with_dendro(
    resp_std,
    kind=Correlation.PEARSON,
    no_cbar=True,
    x_categories=categories,
    y_categories=categories,
)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 15), constrained_layout=True)
plot_corr(resp_std, kind=Correlation.PEARSON, ax=ax);

In [ ]:
fig, ax = plt.subplots(figsize=(15, 15), constrained_layout=True)
plot_corr(resp_std, kind=Correlation.PARTIAL, ax=ax);

In [ ]:
fig, ax = plt.subplots(figsize=(15, 15), constrained_layout=True)
plot_corr(resp_std, kind=Correlation.VAR_TEMPORAL, ax=ax);

In [ ]:
fig, ax = plt.subplots(figsize=(15, 15), constrained_layout=True)
plot_corr(resp_std, kind=Correlation.VAR_CONTEMPORANEOUS, ax=ax);

In [ ]:
def plot_corr_with_dendro(
    df: pl.DataFrame,
    kind: Correlation = Correlation.PEARSON,
    dendro_method: str = "ward",
    categories: list[str] | None = None,
    y_vars: list[str] | None = None,
    figsize: tuple[int, int] | tuple[float, float] | None = None,
    no_cbar: bool = False,
):
    corr = kind.calculate(df)

    # Remove survey metadata columns
    found_cols = []
    for col in ("wave", "participant_id"):
        if col in df.columns:
            found_cols.append(col)
    if found_cols:
        df = df.clone().drop(*found_cols)

    x_labels = np.asarray(df.columns)
    y_labels = np.asarray(df.columns)
    if categories:
        x_categories = np.asarray(categories)
        y_categories = np.asarray(categories)

    # If y_vars specified, select the required rows
    if y_vars is not None:
        keep_idxes = [i for i, label in enumerate(y_labels) if label in y_vars]
        if len(keep_idxes) != len(y_vars):
            raise RuntimeError("Could not find one or more y_var columns in df.")

        corr = corr[keep_idxes]
        y_labels = y_labels[keep_idxes]
        if categories:
            y_categories = y_categories[keep_idxes]

    # Determine figure size if not specified.
    # - Each cell needs ~0.25
    # - Labels need ~0.5
    if figsize is None:
        height = corr.shape[0] * 0.25 + 0.5
        width = corr.shape[1] * 0.25 + 0.5
        figsize = (width, height)

    # Generate a custom diverging colormap
    cmap = sns.diverging_palette(20, 230, as_cmap=True)

    if categories:
        category_pal = sns.husl_palette(5, s=0.45)
        category_lut = dict(
            zip(
                map(
                    str,
                    ["Belief", "Experience", "Attitude", "Behaviour", "Demographic"],
                ),
                category_pal,
            )
        )
        y_category_colours = [category_lut[c] for c in y_categories]
        x_category_colours = [category_lut[c] for c in x_categories]
    else:
        y_category_colours = None
        x_category_colours = None

    if no_cbar:
        g = sns.clustermap(
            corr,
            center=0,
            cmap=cmap,
            row_colors=y_category_colours,
            col_colors=x_category_colours,
            vmin=-1,
            vmax=1,
            method=dendro_method,
            dendrogram_ratio=(0.1, 0.2),
            cbar_pos=None,
            linewidths=0.75,
            figsize=figsize,
            fmt=".1f",
            annot=True,
        )
    else:
        g = sns.clustermap(
            corr,
            center=0,
            cmap=cmap,
            row_colors=y_category_colours,
            col_colors=x_category_colours,
            vmin=-1,
            vmax=1,
            method=dendro_method,
            dendrogram_ratio=(0.1, 0.2),
            cbar_pos=(-0.1, 0.32, 0.03, 0.2),
            linewidths=0.75,
            figsize=figsize,
            fmt=".1f",
            annot=True,
        )

    x_labels = x_labels[g.dendrogram_col.reordered_ind]
    y_labels = y_labels[g.dendrogram_row.reordered_ind]

    g.ax_heatmap.set_xticks(
        np.arange(len(x_labels)) + 0.5,
        x_labels,
        rotation=45,
        horizontalalignment="right",
    )
    g.ax_heatmap.set_yticks(np.arange(len(y_labels)) + 0.5, y_labels, rotation=0)

In [ ]:
plot_corr_with_dendro(
    resp_std, kind=Correlation.PEARSON, no_cbar=True, x_categories=categories
);

In [ ]:
import pandas as pd

sns.set_theme()

# Load the brain networks example dataset
df = sns.load_dataset("brain_networks", header=[0, 1, 2], index_col=0)

# Select a subset of the networks
used_networks = [1, 5, 6, 7, 8, 12, 13, 17]
used_columns = df.columns.get_level_values("network").astype(int).isin(used_networks)
df = df.loc[:, used_columns]

# Create a categorical palette to identify the networks
network_pal = sns.husl_palette(8, s=0.45)
network_lut = dict(zip(map(str, used_networks), network_pal))

# Convert the palette to vectors that will be drawn on the side of the matrix
networks = df.columns.get_level_values("network")
network_colors = pd.Series(networks, index=df.columns).map(network_lut)

# Draw the full plot
g = sns.clustermap(
    df.corr(),
    center=0,
    cmap="vlag",
    row_colors=network_colors,
    col_colors=network_colors,
    dendrogram_ratio=(0.1, 0.2),
    cbar_pos=(0.02, 0.32, 0.03, 0.2),
    linewidths=0.75,
    figsize=(12, 13),
)

g.ax_row_dendrogram.remove()

# Closer look at correlation

## High correlation variables (Pearson $\geq 0.75$)

In [ ]:
high_corr_cols, idxes = filter_vars_by_abs_corr(resp_std, 0.75)
resp_std_high_corr = resp.select(*SURVEY_COLS, *high_corr_cols)

In [ ]:
plot_corr_with_dendro(resp_std_high_corr, kind=Correlation.PEARSON)

In [ ]:
plot_corr_with_dendro(resp_std_high_corr, kind=Correlation.PARTIAL)

In [ ]:
plot_corr_with_dendro(resp_std_high_corr, kind=Correlation.VAR_CONTEMPORANEOUS)

In [ ]:
plot_corr_with_dendro(resp_std_high_corr, kind=Correlation.VAR_TEMPORAL, figsize=(3, 3))

## Moderate--high correlation variables (Pearson $\geq 0.5$)

In [ ]:
mod_corr_cols = filter_vars_by_abs_corr(resp_std, 0.5)
resp_std_mod_corr = resp.select(*SURVEY_COLS, *mod_corr_cols)

In [ ]:
plot_corr_with_dendro(resp_std_mod_corr, kind=Correlation.PEARSON)

In [ ]:
plot_corr_with_dendro(resp_std_mod_corr, kind=Correlation.PARTIAL)

In [ ]:
plot_corr_with_dendro(resp_std_mod_corr, kind=Correlation.VAR_CONTEMPORANEOUS)

In [ ]:
plot_corr_with_dendro(resp_std_mod_corr, kind=Correlation.VAR_TEMPORAL)

## Low correlation variables 

### (Pearson $\leq 0.2$)

In [ ]:
low_corr_cols, low_corr_idxes = filter_vars_by_abs_corr(
    resp_std, upper=0.25, predicate="all"
)
categories_low_corr = np.asarray(categories)[low_corr_idxes]

In [ ]:
categories_low_corr

In [ ]:
plot_corr_with_dendro(
    resp_std,
    kind=Correlation.PEARSON,
    y_vars=low_corr_cols,
    x_categories=categories,
    y_categories=categories,
)

In [ ]:
plot_corr_with_dendro(
    resp_std,
    kind=Correlation.PARTIAL,
    y_vars=low_corr_cols,
    x_categories=categories,
    y_categories=categories,
)

In [ ]:
plot_corr_with_dendro(
    resp_std, kind=Correlation.VAR_CONTEMPORANEOUS, y_vars=low_corr_cols
)

In [ ]:
plot_corr_with_dendro(resp_std, kind=Correlation.VAR_TEMPORAL, y_vars=low_corr_cols)

### Partial $\leq 0.15$

In [ ]:
low_corr_cols = filter_vars_by_abs_corr(
    resp_std, upper=0.15, predicate="all", kind=Correlation.PARTIAL
)

In [ ]:
plot_corr_with_dendro(resp_std, kind=Correlation.PEARSON, y_vars=low_corr_cols)

In [ ]:
plot_corr_with_dendro(resp_std, kind=Correlation.PARTIAL, y_vars=low_corr_cols)

In [ ]:
plot_corr_with_dendro(
    resp_std, kind=Correlation.VAR_CONTEMPORANEOUS, y_vars=low_corr_cols
)

In [ ]:
plot_corr_with_dendro(resp_std, kind=Correlation.VAR_TEMPORAL, y_vars=low_corr_cols)

## Moderate partial correlations ($\geq 0.25$)

In [ ]:
mod_partial_corr_cols = filter_vars_by_abs_corr(
    resp_std, lower=0.25, kind=Correlation.PARTIAL, predicate="any"
)
resp_std_mod_partial = resp.select(*SURVEY_COLS, *mod_partial_corr_cols)

In [ ]:
plot_corr_with_dendro(resp_std_mod_partial, kind=Correlation.PEARSON)

In [ ]:
plot_corr_with_dendro(resp_std_mod_partial, kind=Correlation.PARTIAL)

In [ ]:
plot_corr_with_dendro(resp_std_mod_partial, kind=Correlation.VAR_CONTEMPORANEOUS)

In [ ]:
plot_corr_with_dendro(resp_std_mod_partial, kind=Correlation.VAR_TEMPORAL)

## Moderate VAR contemporaneous correlations ($\geq 0.25$)

In [ ]:
mod_contemp_corr_cols = filter_vars_by_abs_corr(
    resp_std, lower=0.25, kind=Correlation.VAR_CONTEMPORANEOUS, predicate="any"
)
resp_std_mod_contemp = resp.select(*SURVEY_COLS, *mod_contemp_corr_cols)

In [ ]:
plot_corr_with_dendro(resp_std_mod_contemp, kind=Correlation.PEARSON)

In [ ]:
plot_corr_with_dendro(resp_std_mod_contemp, kind=Correlation.PARTIAL)

In [ ]:
plot_corr_with_dendro(resp_std_mod_contemp, kind=Correlation.VAR_CONTEMPORANEOUS)

In [ ]:
plot_corr_with_dendro(resp_std_mod_contemp, kind=Correlation.VAR_TEMPORAL)

# EFA

## Parallel analysis: determining number of factors

In [ ]:
rng = np.random.default_rng(202602271618)

true_eigs = pa_true_eigs(resp_std)
rand_eigs = pa_random_eigs(resp_std, repeats=100, rng=rng)

fig, ax = plt.subplots(figsize=(4.5, 2), constrained_layout=True)
ax.plot(np.arange(len(true_eigs)), true_eigs, color="tab:blue", label="Data")
ax.plot(np.arange(len(rand_eigs)), rand_eigs, color="tab:red", label="Random")
ax.set_xticks(np.arange(0, len(true_eigs), 2), np.arange(0, len(true_eigs), 2) + 1)

ax.set_xlim(0, 44)
ax.set_ylim(0, None)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xlabel("Eigenvalue sorted index (decreasing)")
ax.set_ylabel("Eigenvalue")
ax.legend();

In [ ]:
X = resp_std.drop(*SURVEY_COLS).to_numpy()
labels = resp_std.drop(*SURVEY_COLS).columns

n_factor = 3

efa = Factor(X, n_factor=n_factor).fit()

pca = PCA(n_components=n_factor)
pca.fit(X)

fig, axes = plt.subplots(ncols=2, figsize=(6, 10), constrained_layout=True)

cmap = sns.diverging_palette(20, 230, as_cmap=True)

sns.heatmap(
    efa.loadings,  # * np.sqrt(pca.explained_variance_),
    center=0,
    annot=True,
    fmt=".1f",
    linewidths=0.5,
    square=True,
    cmap=cmap,
    cbar_kws={"aspect": 50},
    ax=axes[0],
)

axes[0].set_yticks(np.arange(efa.loadings.shape[0]) + 0.5, labels, rotation=0)
axes[0].set_title("EFA Factor Loading")

sns.heatmap(
    pca.components_.T * np.sqrt(pca.explained_variance_),
    center=0,
    annot=True,
    fmt=".1f",
    linewidths=0.5,
    square=True,
    cmap=cmap,
    cbar_kws={"aspect": 50},
    ax=axes[1],
)

axes[1].set_yticks(np.arange(pca.components_.shape[-1]) + 0.5, labels, rotation=0)
axes[1].set_title("PCA Factor Loading")

In [ ]:
# Generate a custom diverging colormap
cmap = sns.diverging_palette(20, 230, as_cmap=True)

labels = np.asarray(labels)
efa = Factor(X, n_factor=n_factor).fit()


figsize = (0.25 * efa.loadings.shape[0] + 0.5, 0.2 * n_factor + 1.5)

g = sns.clustermap(
    efa.loadings.T,
    center=0,
    cmap=cmap,
    # row_colors=network_colors, col_colors=network_colors,
    vmin=-1,
    vmax=1,
    method="ward",
    dendrogram_ratio=(0, 0.5),
    cbar_pos=None,
    linewidths=0.75,
    figsize=figsize,
    fmt=".1f",
    annot=True,
)

ord_labels = labels[g.dendrogram_col.reordered_ind]

g.ax_heatmap.set_xticks(
    np.arange(efa.loadings.shape[0]) + 0.5,
    ord_labels,
    rotation=45,
    horizontalalignment="right",
);